In [18]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

In [19]:
# Read from central project blueprint config
with open("../config/config.json", "r") as config_file:
    config=json.load(config_file)
raw_data_path=config["file_paths"]["raw_data_path"]
target=config["data_schemas"]["target_variable"]
seed=config["project_metadata"]["random_state"]

In [20]:
# Ingesting Raw Data
df=pd.read_csv(raw_data_path)

In [21]:
# Seperating Features and Target
X_raw=df.drop(target, axis=1)
y_raw=df[target]

# Stratified Train-Test Split
X_train, X_test, y_train, y_test=train_test_split(X_raw, y_raw, test_size=0.2, stratify=y_raw, random_state=seed)

# Initializing features list for dynamic tracking
current_features=X_train.columns.to_list()

In [22]:
# Target Proxy & Constant Leakage Detection
def run_information_leakage_audit(X_train: pd.DataFrame, y_train: pd.Series, leakage_threshold=0.50):
    structural_drops_suggestions = []
    
    # High-Cardinality Strings
    for col in X_train.columns:
        if X_train[col].dtype == 'object' or X_train[col].dtype == 'str' or X_train[col].nunique() == len(X_train):
            if X_train[col].nunique() / len(X_train) > 0.95:
                structural_drops_suggestions.append(col)
                continue
                
        # Strip Zero/Near-Zero Variance Constants
        mode_frequency = X_train[col].value_counts(normalize=True).iloc[0]
        if mode_frequency > 0.98:
            structural_drops_suggestions.append(col)
            continue

    # Leakage Detection Audit
    # Filter features that have not already been eliminated in previous steps
    remaining_cols = [c for c in X_train.columns if c not in structural_drops_suggestions]
    X_audit = X_train[remaining_cols].copy()
    
    # Fast encoding wrapper for text columns to satisfy entropy functions
    for col in X_audit.select_dtypes(include=['object']).columns:
        X_audit[col] = X_audit[col].astype('category').cat.codes

    # Compute mutual information scores to locate overly-dominant variables
    mi_scores = mutual_info_classif(X_audit, y_train, random_state=seed)
    mi_series = pd.Series(mi_scores, index=remaining_cols)
    
    for col, score in mi_series.items():
        if score > leakage_threshold:
            print(f"[FLAG PROXY]: '{col}' contains an aggressive MI score of {score:.2f} with the target. Likely post-event leakage.")
            structural_drops_suggestions.append(col)
    return structural_drops_suggestions

# Isolate structural drops cleanly from the training pipeline
structural_drops = run_information_leakage_audit(X_train, y_train)
active_pool = [c for c in X_train.columns if c not in structural_drops]


# VARIANCE THRESHOLD

num_cols = X_train[active_pool].select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train[active_pool].select_dtypes(exclude=[np.number]).columns.tolist()

# Drop numerical features with low-variance
var_selector = VarianceThreshold(threshold=0.01)
var_selector.fit(X_train[num_cols])
kept_num_cols = [num_cols[i] for i in range(len(num_cols)) if var_selector.get_support()[i]]
dropped_num_vars = [c for c in num_cols if c not in kept_num_cols]

if dropped_num_vars:
    print(f"Dropped continuous elements with zero operational variance: {dropped_num_vars}")


# MULTICOLINEARITY RESOLUTION (SPEARMAN RANK PURGE)

# We use Spearman rank to capture monotonic non-linear locksteps
corr_matrix = X_train[kept_num_cols].corr(method="spearman").abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Programmatically drops overlapping features that push past the 0.85 limit
to_drop_collinear = [col for col in upper_tri.columns if any(upper_tri[col] > 0.85)]
final_num_cols = [c for c in kept_num_cols if c not in to_drop_collinear]

print(f"Dropped redundant collinear elements (>0.85 Spearman): {to_drop_collinear}")

# Isolate the continuous numerical features that survived the collinearity check
continuous_numerical_cols = [c for c in final_num_cols if c not in to_drop_collinear]

robust_scaler_features = []
standard_scaler_features = []

for col in continuous_numerical_cols:
    # Calculate skewness mathematically
    skew_val = X_train[col].skew()
    
    # If skewness is high, flag it for Robust Scaling to handle the outliers gracefully
    if abs(skew_val) > 0.75:
        print(f"[ALERT - OUTLIERS PRESENT]: '{col}' is heavily skewed (Skew: {skew_val:.2f}).")
        print("  -> Action: Routing to RobustScaler (Median/IQR) to neutralize distortions.")
        robust_scaler_features.append(col)
    else:
        print(f"[INFO - STABLE DISTRIBUTION]: '{col}' is normally distributed (Skew: {skew_val:.2f}).")
        print("  -> Action: Routing to standard StandardScaler.")
        standard_scaler_features.append(col)

print("=====================================================\n")

# Save these lists dynamically into your running notebook state
print(f"Features mapped to RobustScaler due to outliers: {robust_scaler_features}")


# INFORMATIONAL HYPOTHESIS EVALUATION (MUTUAL INFORMATION)

final_cat_cols = cat_cols
if cat_cols:
    X_train_cat_encoded = X_train[cat_cols].copy()
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X_train_cat_encoded[cat_cols] = encoder.fit_transform(X_train[cat_cols])

    # Extract distinct non-linear categorization relationships
    mi_scores = mutual_info_classif(X_train_cat_encoded, y_train, random_state=seed)
    mi_series = pd.Series(mi_scores, index=cat_cols).sort_values(ascending=False)

    # Clean out categorical parameters showing near-zero signal contribution (MI <= 0.002)
    final_cat_cols = mi_series[mi_series > 0.002].index.tolist()
    dropped_mi_vars = [c for c in cat_cols if c not in final_cat_cols]
    print(f"Dropped categorical options with insufficient target entropy signal: {dropped_mi_vars}")


# EMBEDDED STRUCTURAL SELECTION (RANDOM FOREST DEPTH WEIGHTS)
filtered_pool = final_num_cols + final_cat_cols
X_train_filtered = X_train[filtered_pool].copy()

if final_cat_cols:
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X_train_filtered[final_cat_cols] = encoder.fit_transform(X_train[final_cat_cols])

# Run a stratified Random Forest with balanced class weights to fight class imbalance variations
rf = RandomForestClassifier(n_estimators=100, random_state=seed, class_weight="balanced")
rf.fit(X_train_filtered, y_train)

importances = pd.Series(rf.feature_importances_, index=filtered_pool).sort_values(ascending=False)
cumulative_importance = importances.cumsum()

# Retain the exact collection of parameters making up the top 95% of impurity reduction splits
production_features = cumulative_importance[cumulative_importance <= 0.95].index.tolist()

# Fallback mechanism to ensure we preserve core traits if threshold compression cuts too aggressively
if not production_features:
    production_features = importances.head(5).index.tolist()

production_num = [c for c in production_features if c in final_num_cols]
production_cat = [c for c in production_features if c in final_cat_cols]

print(f"\n[FINAL ARCHITECTURE STATUS] Feature spaces successfully downselected from {df.shape[1]-1} down to {len(production_features)}.")
print(f"-> Production Scaler Target Array: {production_num}")
print(f"-> Production Categorical Target Array: {production_cat}")

# CENTRAL STATE COUPLING: AUTOMATED SCHEMATIC SYNC

# Update data layout maps inside our deployment json file automatically
config["data_schemas"]["numerical_features"] = production_num
config["data_schemas"]["categorical_features"] = production_cat
config["data_schemas"]["dropped_features"] = list(
    set(df.columns) - set(production_features) - {target}
)
config["preprocessing_parameters"] = {
    "robust_scaler_features": robust_scaler_features,
    "standard_scaler_features": standard_scaler_features
}

with open("../config/config.json", "w") as f:
    json.dump(config, f, indent=4)

print("\n[SUCCESS] Centralized 'config.json' successfully updated with locked production feature schema boundaries.")

Dropped redundant collinear elements (>0.85 Spearman): ['Frequency of use', 'Age']
[ALERT - OUTLIERS PRESENT]: 'Call  Failure' is heavily skewed (Skew: 1.10).
  -> Action: Routing to RobustScaler (Median/IQR) to neutralize distortions.
[ALERT - OUTLIERS PRESENT]: 'Complains' is heavily skewed (Skew: 3.09).
  -> Action: Routing to RobustScaler (Median/IQR) to neutralize distortions.
[ALERT - OUTLIERS PRESENT]: 'Subscription  Length' is heavily skewed (Skew: -1.34).
  -> Action: Routing to RobustScaler (Median/IQR) to neutralize distortions.
[ALERT - OUTLIERS PRESENT]: 'Charge  Amount' is heavily skewed (Skew: 2.58).
  -> Action: Routing to RobustScaler (Median/IQR) to neutralize distortions.
[ALERT - OUTLIERS PRESENT]: 'Seconds of Use' is heavily skewed (Skew: 1.35).
  -> Action: Routing to RobustScaler (Median/IQR) to neutralize distortions.
[ALERT - OUTLIERS PRESENT]: 'Frequency of SMS' is heavily skewed (Skew: 1.96).
  -> Action: Routing to RobustScaler (Median/IQR) to neutralize dis